In [8]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# Load data
pois = pd.read_csv('Dataset.csv')
hexagons = pd.read_csv('Grid.csv')

# Convert coordinate columns to numeric, coercing errors to NaN
pois['lon'] = pd.to_numeric(pois['lon'], errors='coerce')
pois['lat'] = pd.to_numeric(pois['lat'], errors='coerce')

hexagons['left'] = pd.to_numeric(hexagons['left'], errors='coerce')
hexagons['top'] = pd.to_numeric(hexagons['top'], errors='coerce')
hexagons['right'] = pd.to_numeric(hexagons['right'], errors='coerce')
hexagons['bottom'] = pd.to_numeric(hexagons['bottom'], errors='coerce')

# Drop rows with invalid coordinates
pois = pois.dropna(subset=['lon', 'lat'])
hexagons = hexagons.dropna(subset=['left', 'top', 'right', 'bottom'])

# Calculate hexagon centroids
hexagons['centroid_lon'] = (hexagons['left'] + hexagons['right']) / 2
hexagons['centroid_lat'] = (hexagons['top'] + hexagons['bottom']) / 2

# Assign each POI to a hexagon
def assign_hexagon(poi_lon, poi_lat, hexagons):
    mask = (
        (hexagons['left'] <= poi_lon) & (poi_lon <= hexagons['right']) &
        (hexagons['bottom'] <= poi_lat) & (poi_lat <= hexagons['top'])
    )
    match = hexagons[mask]
    return match['id'].values[0] if len(match) > 0 else None

pois['hex_id'] = pois.apply(
    lambda row: assign_hexagon(row['lon'], row['lat'], hexagons), axis=1
)
# print(hexagons)
print("First 10 POIs with assigned hexagon IDs:")
print(pois[['name', 'lon', 'lat', 'hex_id']].head(10))

print("\nTotal POIs:", len(pois))
print("POIs assigned to hexagons:", pois['hex_id'].notna().sum())
print("POIs not assigned:", pois['hex_id'].isna().sum())

First 10 POIs with assigned hexagon IDs:
                      name        lon        lat hex_id
0        مسجد ملّا اسماعیل  54.363166  31.895873   None
1         مسجد شهید ساداتی  54.383829  31.829847   None
2               مسجد حظیره  54.371269  31.897907   None
3            حسینه گازرگاه  54.373771  31.893881   None
4  مسجد دانشگاه علوم پزشکی  54.340252  31.843144   None
5               مسجد ولایت  54.366102  31.826712   None
6                      NaN  54.351083  31.839957   None
7                      NaN  54.382941  31.899845   None
8                      NaN  54.383797  31.902155   None
9            مسجد جامع یزد  54.368511  31.901437   None

Total POIs: 1112
POIs assigned to hexagons: 0
POIs not assigned: 1112


cleaning data and prepare for next step (feature engineering)

The problem is clear: coordinate system mismatch.

POIs: Geographic coordinates (WGS84) — longitude ~54°, latitude ~32°
Hexagons: Projected coordinates (likely UTM or Web Mercator) — values in millions
You need to transform one dataset to match the other’s coordinate system. Here’s the solution using pyproj.

In [10]:
import pandas as pd
from pyproj import Transformer

# Load data
pois = pd.read_csv('Dataset.csv')
hexagons = pd.read_csv('Grid.csv')

# Convert to numeric
pois['lon'] = pd.to_numeric(pois['lon'], errors='coerce')
pois['lat'] = pd.to_numeric(pois['lat'], errors='coerce')
hexagons['left'] = pd.to_numeric(hexagons['left'], errors='coerce')
hexagons['top'] = pd.to_numeric(hexagons['top'], errors='coerce')
hexagons['right'] = pd.to_numeric(hexagons['right'], errors='coerce')
hexagons['bottom'] = pd.to_numeric(hexagons['bottom'], errors='coerce')

# Drop NaN
pois = pois.dropna(subset=['lon', 'lat'])
hexagons = hexagons.dropna(subset=['left', 'top', 'right', 'bottom'])

# Transform POI coordinates from WGS84 to Web Mercator (EPSG:3857)
# If your hexagons use a different projection (e.g., UTM Zone 40N for Iran), 
# replace 3857 with the appropriate EPSG code
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
pois['x'], pois['y'] = transformer.transform(pois['lon'].values, pois['lat'].values)

# Calculate hexagon centroids
hexagons['center_x'] = (hexagons['left'] + hexagons['right']) / 2
hexagons['center_y'] = (hexagons['top'] + hexagons['bottom']) / 2

# Assign hexagons
def assign_hexagon(row):
    x, y = row['x'], row['y']
    mask = (
        (x >= hexagons['left']) & 
        (x <= hexagons['right']) & 
        (y >= hexagons['bottom']) & 
        (y <= hexagons['top'])
    )
    candidates = hexagons[mask]
    
    if len(candidates) == 0:
        return None
    
    distances = ((candidates['center_x'] - x)**2 + (candidates['center_y'] - y)**2)**0.5
    return candidates.loc[distances.idxmin(), 'id']

pois['hex_id'] = pois.apply(assign_hexagon, axis=1)

# Results
print("First 10 POIs with assigned hexagons:")
print(pois[['name', 'lon', 'lat', 'x', 'y', 'hex_id']].head(10))
print(f"\nTotal POIs: {len(pois)}")
print(f"Assigned: {pois['hex_id'].notna().sum()}")
print(f"Unassigned: {pois['hex_id'].isna().sum()}")

pois.to_csv('POIs_with_hexagons.csv', index=False)
print("\nSaved to POIs_with_hexagons.csv")


First 10 POIs with assigned hexagons:
                      name        lon        lat             x             y  \
0        مسجد ملّا اسماعیل  54.363166  31.895873  6.051680e+06  3.749650e+06   
1         مسجد شهید ساداتی  54.383829  31.829847  6.053980e+06  3.740996e+06   
2               مسجد حظیره  54.371269  31.897907  6.052582e+06  3.749917e+06   
3            حسینه گازرگاه  54.373771  31.893881  6.052861e+06  3.749389e+06   
4  مسجد دانشگاه علوم پزشکی  54.340252  31.843144  6.049129e+06  3.742738e+06   
5               مسجد ولایت  54.366102  31.826712  6.052007e+06  3.740585e+06   
6                      NaN  54.351083  31.839957  6.050335e+06  3.742321e+06   
7                      NaN  54.382941  31.899845  6.053881e+06  3.750171e+06   
8                      NaN  54.383797  31.902155  6.053977e+06  3.750474e+06   
9            مسجد جامع یزد  54.368511  31.901437  6.052275e+06  3.750380e+06   

    hex_id  
0  17205.0  
1  19432.0  
2  17983.0  
3  18297.0  
4  15055.0  
5  

Investigating the 7 unassigned POIs

In [ ]:
print(pois[pois['hex_id'].isna()][['name', 'lon', 'lat']])

                        name        lon        lat
69   مجموعه ورزشی شهید نصیری  54.328391  31.858753
152                      NaN  54.313098  31.833249
230             ورزشگاه معلم  54.382191  31.920325
233   میدان میوه تره بار یزد  54.366374  31.792575
247                      NaN  54.328791  31.858544
356                      NaN  54.328777  31.858555
382            خانه ملک زاده  54.369065  31.904537


In [13]:
# Pivot: count each fclass per hexagon
poi_counts = pois.groupby(['hex_id', 'fclass']).size().unstack(fill_value=0)

# Rename for clarity
poi_counts.columns = [f'count_{col}' for col in poi_counts.columns]
poi_counts = poi_counts.reset_index()

# Display results
print("POI counts by hexagon and fclass:")
print(poi_counts.head(20))
print(f"\nShape: {poi_counts.shape}")
print(f"\nColumns: {list(poi_counts.columns)}")
print(f"\nTotal hexagons with POIs: {len(poi_counts)}")


POI counts by hexagon and fclass:
     hex_id  count_artwork  count_atm  count_attraction  count_bakery  \
0   12253.0              0          0                 0             0   
1   12409.0              0          0                 0             0   
2   13336.0              0          0                 0             0   
3   13494.0              0          0                 0             0   
4   13648.0              0          0                 0             0   
5   13799.0              0          0                 0             0   
6   13801.0              0          0                 0             0   
7   13955.0              0          0                 0             0   
8   13957.0              0          0                 0             0   
9   13959.0              0          0                 0             0   
10  14110.0              0          0                 0             0   
11  14114.0              0          0                 0             0   
12  14267.0      